# project_07_rbd_binder — all notebooks (00→05) in one

This is a **convenience copy** that concatenates the six standalone notebooks in order so you can run the whole project top-to-bottom in a single Colab session. The individual notebooks (`00_setup.ipynb` … `05_validation_plan.ipynb`) remain in this folder and are the canonical deliverables. Sections are separated by dividers; each section keeps its own setup/`import` cells (re-running them is harmless). All synthetic numbers are still labeled `EXAMPLE_DATA`.

---

## ▶︎ Section 1 / 6 — `00_setup.ipynb`

---

# 00 · Environment Setup — De Novo Protein Design Capstone

This is the **shared setup notebook** every project starts from. Run it top to bottom
*once per Colab session*. It:

1. detects your GPU and warns if you're on a weak/absent one,
2. installs a light, pinned core toolset (Biopython, py3Dmol, foldseek-less utilities),
3. optionally installs heavier tools (ColabFold, ESMFold) on demand,
4. prints exact versions for your `LOG.md` (reproducibility is graded).

> **Compute reality.** A free Colab **T4** runs ColabFold, ESMFold, ProteinMPNN, and small
> RFdiffusion jobs. **BindCraft / RFantibody / large RFdiffusion** want an **A100** (Colab Pro+
> or a cluster). Each project's `MANUAL.md` states its tier. Don't fight a T4 to do an A100 job —
> plan your batch sizes around it.

## 1 · GPU & environment check

In [ ]:
import subprocess, sys, platform, textwrap

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

print("Python :", sys.version.split()[0])
print("Platform:", platform.platform())

gpu = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null")
if gpu:
    print("GPU    :", gpu)
    name = gpu.lower()
    if "t4" in name:
        print(textwrap.fill(
            "NOTE: T4 detected. Good for ColabFold/ESMFold/ProteinMPNN/small RFdiffusion. "
            "For BindCraft/RFantibody/large diffusion, switch to A100 (Colab Pro+) or a cluster.", 88))
    elif any(x in name for x in ("a100", "l4", "v100")):
        print("NOTE: capable GPU — heavier tools (BindCraft/RFantibody) are feasible.")
else:
    print("GPU    : NONE FOUND")
    print(textwrap.fill(
        "WARNING: No GPU. Go to Runtime → Change runtime type → Hardware accelerator → GPU. "
        "Structure prediction on CPU is impractically slow.", 88))

## 2 · Pinned core install (fast, T4-friendly)

These are light and used across every project. Pins are conservative; bump them in your repo if needed and **log it**.

In [ ]:
# Core utilities used in every project. Quiet + pinned.
%pip -q install biopython==1.84 py3Dmol==2.4.0 numpy pandas matplotlib seaborn tqdm requests 2>/dev/null
print("Core install done.")

In [ ]:
# Version stamp — copy this block's output into your LOG.md for reproducibility.
import importlib, datetime
mods = ["Bio", "py3Dmol", "numpy", "pandas", "matplotlib", "seaborn", "tqdm", "requests"]
print("# Environment stamp", datetime.datetime.utcnow().isoformat(timespec="seconds"), "UTC")
for m in mods:
    try:
        v = importlib.import_module(m).__version__
    except Exception:
        v = "n/a"
    print(f"{m:14s} {v}")

## 3 · Heavy tools — install *on demand*

Don't install these unless your project needs them this session (they're slow to set up).
Each is wrapped in a function so you only pay the cost when you call it.

In [ ]:
def install_colabfold():
    """ColabFold (AF2). ~3–5 min on first install. T4 OK."""
    import subprocess
    subprocess.run("pip -q install 'colabfold[alphafold-minus-jax]'", shell=True)
    # On Colab, the standard route is the localcolabfold installer or the ColabFold notebook;
    # here we expose the pip route. If it fails, fall back to the official ColabFold notebook
    # and import your sequences. Log whichever path you used.
    print("ColabFold install attempted. Verify with: from colabfold.batch import run")

def install_esmfold():
    """ESMFold via HuggingFace transformers. T4 OK for <~400 aa."""
    import subprocess
    subprocess.run("pip -q install 'transformers>=4.40' accelerate", shell=True)
    print("ESMFold deps installed. Load with transformers EsmForProteinFolding.")

print("Helpers ready: install_colabfold(), install_esmfold().")

## 4 · Reproducibility helpers

Call `set_seeds()` at the top of every run, and use `log()` to append to your `LOG.md`.

In [ ]:
import os, random
import numpy as np

def set_seeds(seed: int = 0):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass
    print(f"seeds set to {seed}")

def log(msg: str, path: str = "LOG.md"):
    import datetime
    stamp = datetime.datetime.utcnow().isoformat(timespec="seconds")
    with open(path, "a") as fh:
        fh.write(f"- {stamp}Z · {msg}\n")
    print("logged:", msg)

set_seeds(0)
log("Ran 00_setup; environment stamped.")

## 5 · (Optional) Mount Google Drive for persistence

Colab sessions are ephemeral. Mount Drive to keep your `results/` and design pools between sessions.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# WORKDIR = "/content/drive/MyDrive/denovo_capstone/project_XX"
# import os; os.makedirs(WORKDIR, exist_ok=True); os.chdir(WORKDIR)
print("Uncomment to mount Drive and set your working directory.")

---
**Next:** open `01_define_and_explore.ipynb`. Keep this session alive — re-running `00_setup`
each new session is normal. Record every version and seed in `LOG.md`.

---

## ▶︎ Section 2 / 6 — `01_define_and_explore.ipynb`

---

# 01 · Define & Explore — the conserved RBD epitope + the binder metrics

**Standard slot:** *define & explore.* **For Project 07 this means:** clean the RBD target, pick a
**conserved, ACE2-facing (neutralizing) epitope**, fix the metrics, and run a tiny mock binder
batch as your hello-world (D0).

> **Defensive framing.** Every design here is steered to *block* the virus at the ACE2 face. Enhancing
> viral affinity/escape/fitness is out of scope (`README.md` → Responsible research, `MASTER_BLUEPRINT.md §7`).

Run `00_setup.ipynb` first in this session.

## The metrics, precisely (binder design)

| Metric | Means | Does **not** mean |
|--------|-------|-------------------|
| `pae_interaction` (AF2-Multimer) | interface confidence — the key binder metric (lower = better) | measured affinity |
| interface pLDDT | local confidence at the interface | stability / K_D |
| scRMSD | designed-vs-predicted binder backbone self-consistency | binding |
| shape complementarity | packing across the interface | function |
| **worst-case breadth pae** | does the binder hold across ALL variant RBDs? | best-case is not breadth |


## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Target prep + epitope choice (EXAMPLE — verify on the real structure)

Clean the RBD (chain selection, remove ACE2), then choose hotspots on the **conserved ACE2-binding
face**. The residues below are **EXAMPLE placeholders** — derive the real ones from 6M0J + a
sarbecovirus conservation analysis (see `data/README.md`).

In [ ]:
import binder_tools as bt

TARGET = "RBD"
# EXAMPLE conserved-epitope hotspots (SARS-CoV-2 numbering) — VERIFY from 6M0J + conservation.
HOTSPOTS = bt.parse_hotspots("E417,E484,E501")
print("target:", TARGET, "| EXAMPLE hotspots (verify):", HOTSPOTS)

## Hello-world: a tiny mock binder batch

Develop the plumbing with the deterministic `mock` backend (no GPU). Switch `tool=` to the real
backends on an A100 (see `MANUAL.md §2`). **Mock numbers are SYNTHETIC — never report them.**

In [ ]:
designs = bt.generate_binders_bindcraft(TARGET, HOTSPOTS, n=5, tool="mock")
bt.score_designs(designs, tool="mock")
d = designs[0]
print("example:", d.design_id, "len=", d.length,
      "pae_interaction=", d.pae_interaction, "scrmsd=", d.scrmsd, "sc=", d.shape_complementarity)
print("ACE2-footprint overlap (competition proxy):", bt.hotspot_overlap(d.contact_residues, HOTSPOTS))
print("[reminder] every number above is SYNTHETIC (mock).")

## Breadth concept: epitope conservation across variants `[core]`

A neutralizing binder is variant-resistant only if its epitope is **conserved**. The snippet below is
**EXAMPLE_DATA** (toy aligned RBD fragments) to demonstrate `epitope_conservation()`; in your project
you align a real variant panel (e.g. with MAFFT) and score the columns under your epitope.

In [ ]:
# EXAMPLE_DATA — toy aligned RBD fragments (NOT real sequences), positions 1..10 for illustration.
EXAMPLE_VARIANTS = {
    "Wuhan":   "NITNLCPFGE",
    "Delta":   "NITNLCPFGE",
    "Omicron": "NITNLCPFGK",   # a change at position 10
}
conserved_epitope = [3, 5, 7]   # toy positions; high conservation
variable_epitope  = [10]        # toy position; low conservation
print("conserved-epitope conservation:", bt.epitope_conservation(conserved_epitope, EXAMPLE_VARIANTS))
print("variable-epitope conservation: ", bt.epitope_conservation(variable_epitope, EXAMPLE_VARIANTS))
print("[EXAMPLE_DATA] toy demonstration of the breadth rationale — replace with the real panel.")

## D0 checklist
- [ ] Conserved-epitope map (conservation across variants) + justification of the chosen face.
- [ ] One-paragraph definition of each binder metric **with** its 'does not mean' note.
- [ ] One reproduced mock mini-run (designs scored, breadth rationale shown).
- [ ] `LOG.md` entry: tool version, GPU, seed.

**Next:** `02_generate.ipynb` — the two-paradigm campaign at the conserved epitope.

---

## ▶︎ Section 3 / 6 — `02_generate.ipynb`

---

# 02 · Campaign — BindCraft + RFdiffusion at the conserved epitope

**Standard slot:** *design campaign.* **For Project 07 this means:** run both binder paradigms against
the conserved ACE2-facing epitope and assemble the pool (D2). Diversity before filtering: generate
hundreds, filter later. The real campaign wants an **A100** — develop on `mock` here.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Verify the upstreams still exist (pin commits — they change)

Tools move; pin a commit and confirm the repo/notebook still resolves before a real run.

In [ ]:
import requests
UPSTREAMS = {
    "BindCraft":     "https://github.com/martinpacesa/BindCraft",       # pin a commit
    "FreeBindCraft": "https://github.com/cytokineking/FreeBindCraft",   # free-tier fallback (VERIFY)
    "RFdiffusion":   "https://github.com/RosettaCommons/RFdiffusion",
    "ColabDesign":   "https://github.com/sokrypton/ColabDesign",
    "ColabFold":     "https://github.com/sokrypton/ColabFold",          # AF2-Multimer
}
for name, url in UPSTREAMS.items():
    try:
        r = requests.head(url, timeout=20, allow_redirects=True)
        print(f"{name:14s} {r.status_code}  {url}")
    except Exception as e:
        print(f"{name:14s} ERR  {url}  ({e})")

## Run both paradigms (mock; switch tool= on an A100)

BindCraft (50–200 designs) + RFdiffusion binder mode (500–1000 backbones → ProteinMPNN). Here we use
small mock counts so the notebook runs anywhere. All numbers are **SYNTHETIC**.

In [ ]:
import binder_tools as bt, pandas as pd

TARGET = "RBD"
HOTSPOTS = bt.parse_hotspots("E417,E484,E501")   # EXAMPLE — verify

bc = bt.generate_binders_bindcraft(TARGET, HOTSPOTS, n=50, tool="mock")     # -> 50-200 real
rf = bt.generate_binders_rfdiffusion(TARGET, HOTSPOTS, n=200, tool="mock")  # -> 500-1000 real
pool = bc + rf
bt.score_designs(pool, tool="mock")
df = pd.DataFrame([d.as_row() for d in pool])
df.to_csv("results/designs.csv", index=False)
print("pool:", len(pool), "| BindCraft:", len(bc), "| RFdiffusion:", len(rf))
print("wrote results/designs.csv", df.shape, "(all SYNTHETIC / EXAMPLE_DATA)")
df.head(3)

## D2 checklist
- [ ] Full pool from **both** paradigms in `results/designs.csv` (hundreds at real scale).
- [ ] Design log: tool versions, params, seeds, runtimes in `LOG.md`.
- [ ] Interim report (3–4 pages).

**Next:** `03_filter_and_rank.ipynb` — the shared binder filter.

---

## ▶︎ Section 4 / 6 — `03_filter_and_rank.ipynb`

---

# 03 · Filter & Rank — the shared multi-layer binder filter

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25 projects
use. **For Project 07** we run the **binder** cutoffs on the pool and report honest survival (D3 pt 1).

Run `00`–`02` first so `results/designs.csv` exists.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Load the shared filtering pipeline

In [ ]:
import filtering_pipeline as fp
import pandas as pd
print("binder cutoffs:", fp.DEFAULT_CUTOFFS["binder"])

## Build `fp.Design` objects from the campaign

Map each binder onto the shared `Design` record (`design_type="binder"`). Layer 2 (orthogonal) needs a
*second* predictor (ESMFold/Boltz) — we run layers **(1, 3)** here and note that L2 is added once you
wire in a second predictor in a real run.

In [ ]:
df = pd.read_csv("results/designs.csv")
designs = [fp.Design(design_id=str(r.design_id), sequence=str(r.sequence), design_type="binder",
                     plddt=r.plddt, pae_interaction=r.pae_interaction, scrmsd=r.scrmsd,
                     shape_complementarity=r.shape_complementarity,
                     extra={"paradigm": r.paradigm, "synthetic": True})
           for r in df.itertuples()]
print(len(designs), "Design objects built (design_type='binder')")

## Run the pipeline + report

In [ ]:
ranked = fp.run_pipeline(designs, design_type="binder", use_layers=(1, 3))
ranked.to_csv("results/ranked.csv", index=False)
top = fp.report(ranked, top_n=10, save_prefix="results/proj07")
print("\nNOTE: numbers are SYNTHETIC (mock). Layer 2 (orthogonal ESMFold/Boltz) is added in a real run.")
top

## Survival-at-each-layer (honest accounting)
Report N pass / N generated at each layer, per paradigm — the honest hit-rate view.

In [ ]:
import pandas as pd
if "layers_passed" in ranked:
    print(ranked["layers_passed"].value_counts().sort_index())

## D3 (part 1) checklist
- [ ] `results/ranked.csv` produced by the **shared** module (`design_type='binder'`).
- [ ] Survival-at-each-layer reported (honest hit rate).
- [ ] Mapping assumptions written down.

**Next:** `04_validate.ipynb` — cross-variant breadth + head-to-head.

---

## ▶︎ Section 5 / 6 — `04_validate.ipynb`

---

# 04 · Validate — cross-variant breadth + head-to-head

**Standard slot:** *validate (in silico).* **For Project 07 the core science is BREADTH:** does each top
binder hold across a panel of variant RBDs? Report the **worst-case** variant, not the best (D3 pt 2).

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Cross-variant breadth

Model each top candidate against the variant panel and record per-variant `pae_interaction`. A broad
neutralizer keeps **all** variants low. The panel below is **EXAMPLE_DATA** (mock); use your real
verified variant RBDs in the project.

In [ ]:
import binder_tools as bt, pandas as pd, numpy as np

ranked = pd.read_csv("results/ranked.csv")
top = ranked.head(10)
VARIANTS = ["Wuhan", "Delta", "Omicron_BA1", "Omicron_XBB", "SARS1"]   # EXAMPLE panel — verify/replace
rows = []
for r in top.itertuples():
    prof = bt.breadth_across_variants(str(r.sequence), VARIANTS, tool="mock")
    prof["design_id"] = r.design_id
    prof["worst_case_pae"] = max(v for v in prof.values() if isinstance(v, (int, float)))
    rows.append(prof)
breadth = pd.DataFrame(rows).set_index("design_id")
breadth = breadth.sort_values("worst_case_pae")   # broadest first (lowest worst-case)
breadth.to_csv("results/breadth.csv")
print("breadth profile (SYNTHETIC) — broadest (lowest worst-case pae) first:")
breadth

In [ ]:
import matplotlib.pyplot as plt
panel = [c for c in breadth.columns if c not in ("worst_case_pae",)]
fig, ax = plt.subplots(figsize=(7, 4))
for did, row in breadth.iterrows():
    ax.plot(panel, [row[c] for c in panel], marker="o", alpha=0.6, label=str(did)[:18])
ax.set_ylabel("pae_interaction (lower = better)"); ax.set_xlabel("variant RBD")
ax.set_title("Cross-variant breadth (EXAMPLE_DATA / mock)")
ax.axhline(10, ls="--", c="k", lw=0.8, label="binder cutoff 10")
plt.xticks(rotation=30); plt.tight_layout(); plt.savefig("results/proj07_breadth.png", dpi=150); plt.show()

## 2 · Head-to-head: BindCraft vs RFdiffusion `[core]`

Compare the two paradigms on survival and interface confidence (honest hit-rate accounting).

In [ ]:
designs = pd.read_csv("results/designs.csv")
by = designs.groupby("paradigm").agg(n=("design_id", "size"),
        mean_pae=("pae_interaction", "mean"), mean_sc=("shape_complementarity", "mean"))
print("per-paradigm summary (SYNTHETIC):")
print(by)
print("\n[reminder] all numbers are mock/EXAMPLE_DATA — compare paradigms on YOUR real campaign.")

## D3 (part 2) checklist
- [ ] Cross-variant breadth table + figure; **worst-case** variant reported per candidate.
- [ ] Conserved- vs variable-epitope comparison (breadth difference).
- [ ] BindCraft vs RFdiffusion head-to-head (hit rate, interface confidence, novelty).
- [ ] Failure-mode notes: which candidates are narrow/escape-prone, and why.

**Next:** `05_validation_plan.ipynb` — the controlled, biosafe validation + breadth-testing plan.

---

## ▶︎ Section 6 / 6 — `05_validation_plan.ipynb`

---

# 05 · Validation + breadth-testing plan (with biosafety oversight)

**Standard slot:** *validation plan.* **For Project 07 this means:** turn the top, broadest candidates
into a costed, controlled wet-lab plan — binding, ACE2-competition, and **pseudovirus** neutralization
across the variant panel — under institutional biosafety oversight (D4/D5).

> **Defensive / biosafe.** Neutralization is tested with a **pseudovirus surrogate (standard BSL-2)** under
> **IBC approval** — never authentic-virus gain-of-function. See `MANUAL.md §7` and `MASTER_BLUEPRINT.md §7`.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Assemble the plan

The plan below is generated from your top breadth candidates and written to `results/validation_plan.md`.
Fill in costs/quantities for your reagents and your institution's IBC pathway.

In [ ]:
import pandas as pd, os
breadth = pd.read_csv("results/breadth.csv") if os.path.exists("results/breadth.csv") else None
top_ids = list(breadth['design_id'][:5]) if breadth is not None else ['<top candidates from nb04>']
plan = f'''# Project 07 — Validation + breadth-testing plan (DRAFT)

## Candidates (broadest by worst-case pae; SYNTHETIC ids in this dry run)
{top_ids}

## Tiered experiments (each with controls)
1. Express + purify (E. coli BL21(DE3)); QC by SDS-PAGE + SEC.            [go/no-go]
2. SPR / BLI vs RBD: K_D, kinetics.                                       [binding]
3. ACE2-competition assay: does the binder block ACE2-RBD?               [mechanism = neutralization proxy]
4. Pseudovirus neutralization across the variant panel (IC50 per variant). [breadth; BSL-2 surrogate, IBC-approved]

## Controls (mandatory)
- Positive: a known neutralizing binder/nanobody.
- Negative: scrambled-interface variant of each candidate.
- Negative: an irrelevant-antigen binder (specificity).

## Breadth read-out
- Report IC50 for EACH variant; rank candidates by WORST-CASE variant, not best.

## Biosafety / responsible research
- Pseudovirus assays + any viral material: institutional biosafety committee (IBC) approval at the
  appropriate containment level. Gene synthesis via an IGSC biosecurity-screening provider.
- Defensive/neutralizing purpose only; no enhancement of viral fitness/affinity/escape (MASTER_BLUEPRINT §7).

## Cost + timeline
- [fill in reagent costs, gene-synthesis quote, instrument time, and a Gantt for ~8-12 weeks of wet lab].
'''
open('results/validation_plan.md', 'w').write(plan)
print('wrote results/validation_plan.md')
print(plan[:900])

## (Stretch) Boltz-2 relative affinity `[stretch]`

If you run Boltz-2 on the top complexes, use its score as a **relative rank** of the candidates — never
as a measured K_D. Report the ranking + caveats; the SPR experiment is what produces a real K_D.

In [ ]:
print("Stretch: wire Boltz-2 affinity as a RELATIVE rank only (never a K_D). See MANUAL.md.")

## D4 / D5 checklist
- [ ] `results/validation_plan.md` completed: tiered experiments + controls + breadth read-out + costs.
- [ ] Biosafety/IBC pathway named; gene-synthesis screening noted; defensive framing explicit.
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release.

You're done — a controlled, breadth-aware, defensively-framed neutralizing-binder campaign.